[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_01/MFM_ch1_risk_measures/MFM_ch1_risk_measures.ipynb)

# MFM_ch1_risk_measures

Risk-adjusted performance across markets: underwater (drawdown) curves, CAGR, volatility, Sharpe ratios on simple returns with their i.i.d. (Lo) standard errors, Sortino, maximum drawdown and Calmar ratios.

*Modelling Financial Markets (MFM), Chapter 1: Stylised Facts and Returns. Author: Daniel Traian Pele.*

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'
Teal     = '#17A2B8'

ASSETS = ['sp500', 'bettr', 'btc', 'eurron', 'gold']
COLORS = {'sp500': MainBlue, 'bettr': IDAred, 'btc': Amber, 'eurron': Forest, 'gold': Purple}
PERIODS = {'sp500': 252, 'bettr': 252, 'btc': 365, 'eurron': 252, 'gold': 252}


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# copia locala a folderului data/market, daca notebook-ul ruleaza din repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')

SOURCES = {
    'sp500':  ('GSPC.INDX',    'close', '1990-01-01'),
    'bettr':  ('BETTR.INDX',   'close', '2014-09-01'),
    'btc':    ('BTC-USD.CC',   'close', '2014-09-17'),
    'eurron': ('REF:EUR',      'close', '2005-07-01'),   # de la introducerea leului nou (RON), 1 iulie 2005
    'gold':   ('XAUUSD.FOREX', 'close', '1990-01-01'),
}

LABELS = {'sp500': 'S&P 500', 'bettr': 'BET-TR (Romania)', 'btc': 'Bitcoin',
          'eurron': 'EUR/RON', 'gold': 'Gold'}

# cotatii OTC (aur): sesiunile partiale de sambata/duminica rup randamentul vineri -> luni,
# deci pastram doar zilele lucratoare (aceeasi conventie ca in Capitolul 0)
WEEKDAYS_ONLY = {'gold'}

_CACHE = {}


def read_market(symbol):
    """Citeste data/market/<SIMBOL>.csv local sau din repo-ul GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    df = pd.read_csv(src, parse_dates=['date']).set_index('date').sort_index()
    df.index.name = 'Date'
    return df


def read_reference_rate(currency='EUR', start='2005-07-01', end='2026-09-18'):
    """Cursul oficial de referinta RON (arhive XML anuale BNR)."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    import re
    import urllib.request
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    df = pd.DataFrame(rows, columns=['Date', 'Close']).drop_duplicates('Date').set_index('Date')
    df.index = pd.to_datetime(df.index)
    _CACHE[key] = df.sort_index().loc[start:end]
    return _CACHE[key]


def load_data(name='sp500'):
    """OHLCV zilnic cu coloane Open/High/Low/Close/Volume (Close = pretul folosit)."""
    symbol, col, start = SOURCES[name]
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start)
    d = read_market(symbol).loc[start:]
    if name in WEEKDAYS_ONLY:
        d = d[d.index.dayofweek < 5]
    out = pd.DataFrame({'Open': d['open'], 'High': d['high'], 'Low': d['low'],
                        'Close': d[col], 'Volume': d['volume']})
    return out[out['Close'] > 0].dropna(subset=['Close'])


def load_close(name):
    """Pretul de inchidere, folosit in toate graficele."""
    return load_data(name)['Close'].dropna()


closes = {a: load_close(a) for a in ASSETS}
rets = {a: np.log(c).diff().dropna() for a, c in closes.items()}
# gold (weekdays only) is annualised with its actual number of observations per year
PERIODS['gold'] = len(rets['gold']) / ((closes['gold'].index[-1] - closes['gold'].index[0]).days / 365.25)
spx_ohlc = load_data('sp500')
for a in ASSETS:
    print(f'{LABELS[a]:18s}', rets[a].index[0].date(), '->', rets[a].index[-1].date(), len(rets[a]), 'obs')

## Helper functions

In [ ]:
def drawdown(close):
    """Drawdown procentual fata de maximul anterior."""
    return close / close.cummax() - 1

## Analysis

In [ ]:
def risk_table():
    rows = []
    for a in ASSETS:
        # Sharpe si Sortino pe randamente SIMPLE (aritmetice), r_f = 0: aceeasi definitie in tot cursul
        c, P = closes[a], PERIODS[a]
        R = c.pct_change().dropna()
        years = (c.index[-1] - c.index[0]).days / 365.25
        cagr = (c.iloc[-1] / c.iloc[0]) ** (1 / years) - 1
        mu, vol = R.mean() * P, R.std() * np.sqrt(P)
        downside = np.sqrt((np.minimum(R, 0) ** 2).mean()) * np.sqrt(P)   # medie peste TOATE zilele
        mdd = drawdown(c).min()
        sr_d = R.mean() / R.std()
        rows.append({'asset': LABELS[a], 'CAGR_pct': 100 * cagr, 'mean_simple_pct': 100 * mu,
                     'ann_vol_pct': 100 * vol, 'downside_pct': 100 * downside,
                     'Sharpe': mu / vol, 'Sharpe_SE_Lo': np.sqrt(P * (1 + sr_d ** 2 / 2) / len(R)),
                     'years': years, 'Sortino': mu / downside,
                     'maxDD_pct': 100 * mdd, 'Calmar': cagr / abs(mdd)})
    t = pd.DataFrame(rows).set_index('asset')
    t.to_csv(os.path.join(TABLE_DIR, 'ch1_risk_measures.csv'), float_format='%.6g')
    return t


def fig_drawdowns():
    fig, ax = plt.subplots(figsize=(7.0, 3.0))
    for a in ASSETS:
        dd = drawdown(closes[a])
        ax.plot(dd.index, 100 * dd, color=COLORS[a], lw=0.8, label=f'{LABELS[a]} (max {100 * dd.min():.0f}%)')
    ax.set_ylabel('Drawdown from running peak (%)')
    ax.set_title('Underwater curves: depth and duration of losses', fontsize=9, loc='left')
    legend_outside_bottom(ax, ncol=3, y=-0.12)
    plt.tight_layout()
    save_fig('ch1_drawdowns')


def fig_risk_return(t):
    fig, ax = plt.subplots(figsize=(6.4, 3.0))
    for a in ASSETS:
        row = t.loc[LABELS[a]]
        ax.scatter(row['ann_vol_pct'], row['CAGR_pct'], s=60, color=COLORS[a], zorder=3)
        off = {'sp500': (8, -14), 'gold': (-8, 14), 'bettr': (8, 0), 'btc': (-10, -18), 'eurron': (8, 2)}[a]
        ax.annotate(f"{LABELS[a]}\nSharpe {row['Sharpe']:.2f} (SE {row['Sharpe_SE_Lo']:.2f}), MDD {row['maxDD_pct']:.0f}%",
                    (row['ann_vol_pct'], row['CAGR_pct']), xytext=off, textcoords='offset points', fontsize=7,
                    ha='right' if off[0] < 0 else 'left')
    ax.set_xscale('log')
    ax.set_xticks([5, 10, 20, 50, 100])
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:g}'))
    ax.xaxis.set_minor_formatter(plt.NullFormatter())
    ax.set_xlabel('Annualised volatility (%, log scale)')
    ax.set_ylabel('CAGR (%)')
    ax.set_xlim(3, 120)
    ax.axhline(0, color=Gray, lw=0.4)
    ax.set_title('Risk and return across markets (full sample of each series)', fontsize=9, loc='left')
    plt.tight_layout()
    save_fig('ch1_risk_return')

## Run

In [ ]:
TABLE_DIR = '.'
rt = risk_table()
print(rt.round(3))
fig_drawdowns()
fig_risk_return(rt)

![ch1_drawdowns](./ch1_drawdowns.png)

Output: `ch1_drawdowns.pdf`

![ch1_risk_return](./ch1_risk_return.png)

Output: `ch1_risk_return.pdf`